# MAIA NER — Fine-tune DistilBERT trên Colab T4 (PhoNER_COVID19)

Notebook **tự chứa toàn bộ code** (cells `%%writefile` bên dưới) — bạn **không cần upload gì**.
1. Colab: **Runtime → Change runtime type → T4 GPU**.
2. Chạy từng cell theo thứ tự (Shift+Enter).

## Sau khi chạy xong (cell cuối)
Kết quả được copy về `MAIA-NER/output/` trên Google Drive. Bạn tải về đặt vào project:
- `output/checkpoints/*` → `MAIA/ner/checkpoints/`
- `output/ner-report-*.json` → `MAIA/eval/`
- Gửi con số `micro_f1` để cập nhật `MODEL_CARD.md` (baseline BiLSTM local: 0.8325, paper SOTA PhoBERT-large: 0.945).

In [ ]:
# Cell 1 — Libs + kiểm tra GPU
!pip install -q transformers datasets seqeval accelerate

import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chưa có GPU: Runtime → Change runtime type → T4 GPU rồi chạy lại'


In [ ]:
%%writefile ner/dataset.py
"""PhoNER_COVID19 (word-level, JSONL) -> HF datasets. Source:
https://github.com/VinAIResearch/PhoNER_COVID19 (data/word/*.json, 1 obj/line).
Kept raw here under ner/data/ (gitignored if large; see requirements).
"""
import json
import os

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data")


def load_split(name: str) -> list[dict]:
    path = os.path.join(DATA_DIR, f"{name}_word.json")
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def label_list(rows: list[dict]) -> list[str]:
    labs = set()
    for r in rows:
        labs.update(r["tags"])
    # BIO order: O first, then B-/I- sorted — deterministic id mapping.
    others = sorted(labs - {"O"})
    return ["O"] + others


def to_hf_dataset(rows: list[dict]):
    from datasets import Dataset
    return Dataset.from_dict({
        "words": [r["words"] for r in rows],
        "tags": [r["tags"] for r in rows],
    })


In [ ]:
%%writefile ner/train.py
"""Fine-tune multilingual DistilBERT for Vietnamese NER (PhoNER_COVID19).

Why DistilBERT-multilingual and not PhoBERT: this box is an Intel Mac without
CUDA and PhoBERT needs VnCoreNLP word-segmentation at inference; mDistilBERT
takes raw text (WordPiece) and trains to a legit baseline in CPU time.
Paper reference (reproduced setup, smaller backbone): Nguyen et al. 2020,
"PhoNER_COVID19" — PhoBERT-large F1 0.945; mBERT baseline ~0.90.

Usage: ./ner/.venv/bin/python ner/train.py [--epochs 3] [--max-train 0=all]
Logs Trainer JSON lines to ner/checkpoints/training_log.jsonl
"""
import argparse
import json
import os

import numpy as np

from dataset import label_list, load_split, to_hf_dataset

MODEL_ID = "distilbert-base-multilingual-cased"
OUT_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "checkpoints")


def tokenize_and_align(examples, tokenizer, tag2id, max_len=128):
    tok = tokenizer(examples["words"], truncation=True, is_split_into_words=True,
                    max_length=max_len)
    labels = []
    for i, tags in enumerate(examples["tags"]):
        word_ids = tok.word_ids(batch_index=i)
        prev = None
        ids = []
        for w in word_ids:
            if w is None:
                ids.append(-100)
            elif w != prev:
                ids.append(tag2id[tags[w]])
            else:
                ids.append(-100)
            prev = w
        labels.append(ids)
    tok["labels"] = labels
    return tok


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=float, default=3.0)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--lr", type=float, default=3e-5)
    ap.add_argument("--max-train", type=int, default=0, help="0 = full 5027 sents")
    ap.add_argument("--max-dev", type=int, default=0, help="0 = full 2000 sents")
    ap.add_argument("--max-len", type=int, default=128)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    from transformers import (AutoModelForTokenClassification, AutoTokenizer,
                              DataCollatorForTokenClassification, Trainer,
                              TrainingArguments)

    train_rows = load_split("train")
    dev_rows = load_split("dev")
    if args.max_train:
        train_rows = train_rows[:args.max_train]
    if args.max_dev:
        dev_rows = dev_rows[:args.max_dev]
    labs = label_list(train_rows + dev_rows)
    tag2id = {t: i for i, t in enumerate(labs)}
    print(f"labels({len(labs)}): {labs}")
    print(f"train={len(train_rows)} dev={len(dev_rows)}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    train_ds = to_hf_dataset(train_rows).map(
        lambda ex: tokenize_and_align(ex, tokenizer, tag2id, max_len=args.max_len), batched=True)
    dev_ds = to_hf_dataset(dev_rows).map(
        lambda ex: tokenize_and_align(ex, tokenizer, tag2id, max_len=args.max_len), batched=True)
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_ID, num_labels=len(labs), id2label={i: t for t, i in tag2id.items()},
        label2id=tag2id)

    import transformers
    tr_args = TrainingArguments(
        output_dir=OUT_DIR,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=args.lr,
        per_device_train_batch_size=args.batch,
        per_device_eval_batch_size=args.batch,
        num_train_epochs=args.epochs,
        weight_decay=0.01,
        seed=args.seed,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        disable_tqdm=False,
    )
    trainer = Trainer(model=model, args=tr_args, train_dataset=train_ds,
                      eval_dataset=dev_ds,
                      data_collator=DataCollatorForTokenClassification(tokenizer),
                      tokenizer=tokenizer)
    out = trainer.train()
    trainer.save_model(OUT_DIR)
    with open(os.path.join(OUT_DIR, "label_list.json"), "w") as fh:
        json.dump(labs, fh)
    log = {"model": MODEL_ID, "epochs": args.epochs, "batch": args.batch,
           "lr": args.lr, "max_len": args.max_len,
           "n_train": len(train_rows), "n_dev": len(dev_rows),
           "train_loss": out.training_loss,
           "cpu_budget_note": ("subset + max_len kept small for CPU-only Intel Mac; "
                               "full-data numbers need GPU — see MODEL_CARD.md"),
           "transformers": transformers.__version__}
    with open(os.path.join(OUT_DIR, "training_log.json"), "w") as fh:
        json.dump(log, fh, indent=1)
    print(json.dumps(log, indent=1))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile ner/eval.py
"""Entity-level eval (seqeval P/R/F1) on PhoNER test split + error analysis.
Writes eval/ner-report-<date>.json and prints the table.
Usage: ./ner/.venv/bin/python ner/eval.py [--ckpt ner/checkpoints] [--max-test 0]
"""
import argparse
import json
import os
from datetime import date

import numpy as np


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "checkpoints"))
    ap.add_argument("--max-test", type=int, default=0)
    ap.add_argument("--batch", type=int, default=16)
    args = ap.parse_args()

    import torch
    from seqeval.metrics import classification_report
    from transformers import AutoModelForTokenClassification, AutoTokenizer

    import sys
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    from dataset import load_split

    labs = json.load(open(os.path.join(args.ckpt, "label_list.json")))
    id2lab = {i: t for i, t in enumerate(labs)}
    tokenizer = AutoTokenizer.from_pretrained(args.ckpt)
    model = AutoModelForTokenClassification.from_pretrained(args.ckpt)
    model.eval()

    rows = load_split("test")
    if args.max_test:
        rows = rows[:args.max_test]
    y_true, y_pred = [], []
    with torch.no_grad():
        for i in range(0, len(rows), args.batch):
            chunk = rows[i:i + args.batch]
            tok = tokenizer([r["words"] for r in chunk], truncation=True,
                            is_split_into_words=True, padding=True, max_length=128,
                            return_tensors="pt")
            logits = model(**tok).logits.argmax(-1).tolist()
            for r, ids, wids in zip(chunk, logits,
                                    [tok.word_ids(b) for b in range(len(chunk))]):
                prev, yt, yp = None, [], []
                for lid, w in zip(ids, wids):
                    if w is None or w == prev:
                        prev = w
                        continue
                    prev = w
                    yt.append(r["tags"][w])
                    yp.append(id2lab[lid])
                y_true.append(yt)
                y_pred.append(yp)
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    # per-type F1 for the story
    per_type = {k: round(v["f1-score"], 3) for k, v in rep.items()
                if k not in ("accuracy", "macro avg", "micro avg", "weighted avg")}
    report = {
        "date": str(date.today()),
        "model_ckpt": args.ckpt,
        "n_test": len(rows),
        "micro_f1": round(rep["micro avg"]["f1-score"], 4),
        "macro_f1": round(rep["macro avg"]["f1-score"], 4),
        "precision_micro": round(rep["micro avg"]["precision"], 4),
        "recall_micro": round(rep["micro avg"]["recall"], 4),
        "per_type_f1": per_type,
        "paper_reference": "Nguyen et al. 2020 PhoNER_COVID19: PhoBERT-large 0.945 / mBERT ~0.90 (test micro-F1)",
    }
    out = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))),
                       "eval", f"ner-report-{date.today()}.json")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    json.dump(report, open(out, "w"), ensure_ascii=False, indent=1)
    print(json.dumps(report, ensure_ascii=False, indent=1))
    print(f"wrote {out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile ner/infer.py
"""Inference CLI: raw Vietnamese text -> entities (word-seg tolerant).
Usage: ./ner/.venv/bin/python ner/infer.py --text "Bệnh viện Bạch Mai tiếp nhận bệnh nhân COVID-19"
"""
import argparse
import json
import os


def extract(text: str, ckpt: str, tokenizer, model) -> list[dict]:
    import torch
    words = text.split()
    tok = tokenizer([words], is_split_into_words=True, return_tensors="pt",
                    truncation=True, max_length=128)
    with torch.no_grad():
        ids = model(**tok).logits.argmax(-1)[0].tolist()
    labs = json.load(open(os.path.join(ckpt, "label_list.json")))
    wids = tok.word_ids(batch_index=0)
    ents, prev, cur = [], None, None
    for lid, w in zip(ids, wids):
        if w is None or w == prev:
            prev = w
            continue
        prev = w
        lab = labs[lid]
        if lab.startswith("B-"):
            if cur:
                ents.append(cur)
            cur = {"type": lab[2:], "tokens": [words[w]]}
        elif lab.startswith("I-") and cur and cur["type"] == lab[2:]:
            cur["tokens"].append(words[w])
        else:
            if cur:
                ents.append(cur)
            cur = None
    if cur:
        ents.append(cur)
    return [{"type": e["type"], "text": " ".join(e["tokens"]).replace("_", " ")} for e in ents]


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--text", required=True)
    ap.add_argument("--ckpt", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "checkpoints"))
    args = ap.parse_args()
    from transformers import AutoModelForTokenClassification, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(args.ckpt)
    model = AutoModelForTokenClassification.from_pretrained(args.ckpt)
    model.eval()
    print(json.dumps(extract(args.text, args.ckpt, tokenizer, model), ensure_ascii=False, indent=1))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# Cell 6 — Tải dataset PhoNER gốc từ GitHub VinAI (2.3MB / 1MB / 1.5MB)
!mkdir -p ner/data
!curl -sL -o ner/data/train_word.json https://raw.githubusercontent.com/VinAIResearch/PhoNER_COVID19/main/data/word/train_word.json
!curl -sL -o ner/data/dev_word.json https://raw.githubusercontent.com/VinAIResearch/PhoNER_COVID19/main/data/word/dev_word.json
!curl -sL -o ner/data/test_word.json https://raw.githubusercontent.com/VinAIResearch/PhoNER_COVID19/main/data/word/test_word.json
!ls -la ner/data/ ner/*.py
!head -c 200 ner/data/train_word.json; echo


In [ ]:
# Cell 7 — TRAIN (T4 ~5-10 phút, full 5027 câu, 3 epochs) → ner/checkpoints/
%cd ner
!python train.py --epochs 3 --batch 16
!ls -la checkpoints/


In [ ]:
# Cell 8 — EVAL seqeval P/R/F1 trên test (3000 câu) + thử inference 1 câu
!python eval.py
!python infer.py --text "Bệnh viện Bạch Mai ở Hà Nội tiếp nhận bệnh nhân COVID-19"


In [ ]:
# Cell 9 — Đẩy kết quả về Drive để tải về bỏ vào project
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/MAIA-NER/output'  # <-- sửa nếu muốn chỗ khác
!mkdir -p "$OUT"
!rm -rf "$OUT/checkpoints" "$OUT"/ner-report-*.json
!cp -r checkpoints "$OUT/checkpoints"
!cp /content/eval/ner-report-*.json "$OUT/"
!ls -la "$OUT" "$OUT/checkpoints"
print('Xong. Trên máy local: output/checkpoints/* -> MAIA/ner/checkpoints/, output/ner-report-*.json -> MAIA/eval/')
